In [1]:
import os
import gc
import h5py
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, global_max_pool, global_add_pool, TopKPooling 
from torch_geometric.nn import GATConv, GINEConv, NNConv, SAGEConv, GCNConv
from torch_geometric.nn import BatchNorm, AttentionalAggregation, LayerNorm
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from scipy.signal import hilbert
from scipy.signal import welch
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import copy
from scipy.stats import skew, kurtosis, entropy
from statsmodels.tsa.ar_model import AutoReg
from scipy.fftpack import dct
from scipy.signal import periodogram, butter, filtfilt

/home/ranessa/anaconda3/envs/tf/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
# =========================================================
# CHANNEL MAP
# channel name -> index in the 62-channel SEED EEG layout
# =========================================================

canais_ind = {
    'FP1': 0, 'FPz': 1, 'FP2': 2, 'AF3': 3, 'AF4': 4, 'F7': 5, 'F5': 6, 
    'F3': 7, 'F1': 8, 'Fz': 9, 'F2': 10, 'F4': 11, 'F6': 12, 'F8': 13, 
    'FT7': 14, 'FC5': 15, 'FC3': 16, 'FC1': 17, 'FCz': 18, 'FC2': 19, 
    'FC4': 20, 'FC6': 21, 'FT8': 22, 'T7': 23, 'C5': 24, 'C3': 25, 
    'C1': 26, 'Cz': 27, 'C2': 28, 'C4': 29, 'C6': 30, 'T8': 31, 
    'TP7': 32, 'CP5': 33, 'CP3': 34, 'CP1': 35, 'CPz': 36, 'CP2': 37, 
    'CP4': 38, 'CP6': 39, 'TP8': 40, 'P7': 41, 'P5': 42, 'P3': 43, 
    'P1': 44, 'Pz': 45, 'P2': 46, 'P4': 47, 'P6': 48, 'P8': 49, 
    'PO7': 50, 'PO5': 51, 'PO3': 52, 'POz': 53, 'PO4': 54, 'PO6': 55, 
    'PO8': 56, 'CB1': 57, 'O1': 58, 'Oz': 59, 'O2': 60, 'CB2': 61
}

In [3]:
# =========================================================
# SYMMETRIC CHANNEL PAIRS
# each entry is a [left_hemisphere, right_hemisphere] pair
# used for hemispheric-symmetric channel selection
# pair index is shown as comment for easy reference
# =========================================================

channel_pairs = [
    [canais_ind['FP1'], canais_ind['FP2']], #0
    [canais_ind['AF3'], canais_ind['AF4']], #1

    [canais_ind['F7'],  canais_ind['F8']], #2
    [canais_ind['F5'],  canais_ind['F6']], #3
    [canais_ind['F3'],  canais_ind['F4']], #4
    [canais_ind['F1'],  canais_ind['F2']], #5

    [canais_ind['FT7'], canais_ind['FT8']], #6
    [canais_ind['FC5'], canais_ind['FC6']], #7
    [canais_ind['FC3'], canais_ind['FC4']], #8
    [canais_ind['FC1'], canais_ind['FC2']], #9

    [canais_ind['T7'],  canais_ind['T8']], #10
    [canais_ind['C5'],  canais_ind['C6']], #11
    [canais_ind['C3'],  canais_ind['C4']], #12
    [canais_ind['C1'],  canais_ind['C2']], #13

    [canais_ind['TP7'], canais_ind['TP8']], #14
    [canais_ind['CP5'], canais_ind['CP6']], #15
    [canais_ind['CP3'], canais_ind['CP4']], #16
    [canais_ind['CP1'], canais_ind['CP2']], #17

    [canais_ind['P7'],  canais_ind['P8']], #18
    [canais_ind['P5'],  canais_ind['P6']], #19
    [canais_ind['P3'],  canais_ind['P4']], #20
    [canais_ind['P1'],  canais_ind['P2']], #21

    [canais_ind['PO7'], canais_ind['PO8']], #22
    [canais_ind['PO5'], canais_ind['PO6']], #23
    [canais_ind['PO3'], canais_ind['PO4']], #24
    
    [canais_ind['CB1'], canais_ind['CB2']], #25
    [canais_ind['O1'],  canais_ind['O2']], #26
]


In [4]:
# =========================================================
# DATA LOADING FUNCTION
# =========================================================

def load_selected_data(feature_file, selected_channels, selected_bands):
    """
    Load EEG features from HDF5 file and select specific channels and bands.

    Parameters
    ----------
    feature_file : str
        Path to the HDF5 feature file (output of extracting_DE_features).
    selected_channels : list
        List of channel indices to select.
    selected_bands : list of str
        List of frequency band names to select (e.g. ["delta", "theta"]).

    Returns
    -------
    tuple
        (X, y) where X has shape (N, n_selected_channels, n_windows, n_selected_bands)
        and y has shape (N,).
    """
    with h5py.File(feature_file, "r") as f:
        X = f["data"][:]        # shape: (N, C, W, B)
        y = f["labels"][:]
        all_bands = [b.decode() for b in f["band_names"][:]]

    print("\nOriginal shape:", X.shape)

    # get indices of the selected bands in the full band list
    band_indices = [all_bands.index(b) for b in selected_bands]

    # select channels and bands
    X = X[:, selected_channels, :, :]
    X = X[:, :, :, band_indices]

    print("Shape after selection:", X.shape)

    return X, y



# =========================================================
# CONFIGURATION
# =========================================================

FILE = "EEG_DE_data.h5"  # path to the extracted DE features file

# selected channel pairs: FT7/FT8 (frontotemporal) + P7/P8 (lateral parietal)
channels_selected=channel_pairs[6]+channel_pairs[18]

# channels_selected = list(range(62)) # uncomment to use all 62 channels

# selected frequency bands
bands_selected = [
    "delta",
    # "alpha",
    "theta",
    # "beta",
    # "gamma",
]

# =========================================================
# LOAD DATA
# =========================================================
X_all, y_all = load_selected_data(FILE, channels_selected, bands_selected)

print("\nSummary:")
print("X_all shape:", X_all.shape)   # (N, channels, windows, bands)
print("y_all shape:", y_all.shape)
print("Selected channels:", channels_selected)
print("Selected bands:", bands_selected)


# =========================================================
# SPLIT DATA BY SUBJECT
# each subject has 45 samples (15 trials x 3 sessions)
# =========================================================
samples_per_subject = 45
n_subjects = 15
subject_data = []
subject_labels = []

for s in range(n_subjects):
    start = s * samples_per_subject
    end   = start + samples_per_subject
    subject_data.append(X_all[start:end])
    subject_labels.append(y_all[start:end])

print("\nSubjects separated:", len(subject_data))
print("Subject 0 shape:", subject_data[0].shape)


Original shape: (675, 62, 240, 5)
Shape after selection: (675, 4, 240, 2)

Summary:
X_all shape: (675, 4, 240, 2)
y_all shape: (675,)
Selected channels: [14, 22, 41, 49]
Selected bands: ['delta', 'theta']

Subjects separated: 15
Subject 0 shape: (45, 4, 240, 2)


In [5]:
# =========================================================
# GRAPH CONSTRUCTION
# =========================================================

def build_graphs(eeg_data, eeg_labels):
    """
    Build a list of PyG Data graphs from EEG feature arrays.

    Each sample is represented as a graph where:
    - Nodes correspond to EEG channels
    - Node features are the flattened DE feature vectors (windows x bands)
    - Edges are defined by cosine similarity between node feature vectors,
      retaining only connections above the 70th percentile threshold

    Parameters
    ----------
    eeg_data : np.ndarray
        EEG feature array of shape (N, C, W, B), where N is the number
        of samples, C is the number of channels, W is the number of
        windows, and B is the number of frequency bands.
    eeg_labels : np.ndarray
        Label array of shape (N,).

    Returns
    -------
    list of torch_geometric.data.Data
        List of graph objects, one per sample.
    """

    n_samples, n_channels, n_windows, n_bands = eeg_data.shape
    graphs = []

    for i in range(n_samples):

        # extract single sample: (C, W, B)
        x_sample = eeg_data[i]  

        # flatten time and band dimensions into a single feature vector per node
        # (C, W, B) -> (C, W*B)
        eeg_matrix = x_sample.reshape(n_channels, -1)

        # per-channel z-score normalization to ensure equal contribution
        # to cosine similarity regardless of absolute DE magnitude
        x_norm = (eeg_matrix - eeg_matrix.mean(axis=1, keepdims=True)) / \
                 (eeg_matrix.std(axis=1, keepdims=True) + 1e-8)

        x = torch.tensor(x_norm, dtype=torch.float)

        # compute pairwise cosine similarity between raw feature vectors
        # shape: (C, C)
        similarity = cosine_similarity(eeg_matrix)

        # replace NaN values that may arise from zero-norm vectors
        similarity = np.nan_to_num(similarity)

        # remove self-loops: a node should not be connected to itself
        # (GATConv adds self-loops internally by default)
        np.fill_diagonal(similarity, 0)

        # adaptive sparsification: retain only the top 30% of connections
        # by keeping edges above the 70th percentile of similarity values
        threshold = np.percentile(similarity, 70)

        adjacency = similarity.copy()
        adjacency[adjacency < threshold] = 0

        # build edge list from the sparse adjacency matrix
        edge_index = []

        for u in range(n_channels):
            for v in range(n_channels):
                if adjacency[u, v] > 0:
                    edge_index.append([u, v])

        # fallback: if sparsification removes all edges, use a fully
        # connected graph with uniform weights to ensure message passing
        if len(edge_index) == 0:
            for u in range(n_channels):
                for v in range(n_channels):
                    if u != v:
                        edge_index.append([u, v])
                       

        # convert to PyG-compatible format: shape (2, E)
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        
        # emotion class label for this sample
        y = torch.tensor([eeg_labels[i]], dtype=torch.long)

        # assemble PyG Data object for this sample
        graphs.append(Data(
            x=x,
            edge_index=edge_index,
            y=y
        ))

    return graphs

In [6]:
# =========================================================
# MODEL 
# =========================================================

class EEGGAT(nn.Module):
    """
    Graph Attention Network for EEG-based emotion classification.

    Architecture:
    1. Input projection: Linear + ReLU + LayerNorm
    2. Two GAT convolutional layers with multi-head attention
    3. Dual readout: global mean pooling + attentional aggregation
    4. Classification head: Linear + ReLU + Linear

    Parameters
    ----------
    in_channels : int
        Number of input features per node (W * B).
    hidden_channels : int
        Hidden dimension used throughout the network.
    num_classes : int
        Number of output emotion classes.
    dropout : float
        Dropout rate applied after each GAT layer and in the classifier.
    """
    
    def __init__(self, in_channels, hidden_channels, num_classes, dropout=0.4):
        super().__init__()

        # --- input projection ---
        # projects raw node features into the hidden space
        # LayerNorm is used instead of BatchNorm because with only 4 nodes
        # per graph, batch statistics are insufficiently stable
        self.input_proj = nn.Sequential(
            nn.Linear(in_channels, hidden_channels),
            nn.ReLU(),
            nn.LayerNorm(hidden_channels)
        )

        # --- GAT layers ---
        # first layer: 4 attention heads, outputs averaged (concat=False)
        self.conv1 = GATConv(hidden_channels, hidden_channels, heads=4, concat=False)
        self.bn1 = BatchNorm(hidden_channels)

        # second layer: 2 attention heads, outputs averaged (concat=False)
        self.conv2 = GATConv(hidden_channels, hidden_channels, heads=2, concat=False)
        self.bn2 = BatchNorm(hidden_channels)

        # --- attentional pooling ---
        # learns to weight nodes by relevance to the classification task
        # gate_nn maps each node representation to a scalar score
        self.att_pool = AttentionalAggregation(
            gate_nn=nn.Sequential(
                nn.Linear(hidden_channels, hidden_channels // 2),
                nn.ReLU(),
                nn.Linear(hidden_channels // 2, 1)
            )
        )

        # --- classification head ---
        # input size is hidden_channels * 2 because mean and attentional
        # pooling outputs are concatenated before classification
        self.fc1 = nn.Linear(hidden_channels * 2, hidden_channels)
        self.fc2 = nn.Linear(hidden_channels, num_classes)

        self.dropout = dropout

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # --- input projection ---
        x = self.input_proj(x)

        # --- GAT layers ---
        # each layer: GAT convolution -> BatchNorm -> ReLU -> Dropout
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)

        # --- graph-level readout ---
        # two pooling strategies applied in parallel and concatenated
        x_att = self.att_pool(x, batch)             # attentional aggregation
        x_mean = global_mean_pool(x, batch)         # unweighted mean of all nodes

        # concatenate both representations: shape (batch_size, hidden_channels * 2)
        x = torch.cat([x_mean, x_att], dim=1)

        # --- classification ---
        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)

        # return raw logits; loss function applies softmax internally
        return self.fc2(x)

In [7]:
# =========================================================
# TRAINING UTILITIES
# =========================================================

def compute_train_stats(graphs):
    """
    Compute mean and standard deviation of node features across all
    training graphs, used for feature normalization.

    Parameters
    ----------
    graphs : list of torch_geometric.data.Data
        List of training graphs.

    Returns
    -------
    tuple of torch.Tensor
        (mean, std), each of shape (n_features,).
    """
    # concatenate all node feature matrices across training graphs
    xs = torch.cat([g.x for g in graphs], dim=0)

    # compute per-feature statistics across all training nodes
    mean = xs.mean(dim=0)
    std  = xs.std(dim=0, unbiased=False)

    # replace near-zero std with 1 to avoid division by zero
    std = torch.where(std < 1e-6, torch.ones_like(std), std)

    return mean, std




def augment_batch(batch, edge_drop_prob=0.15, feat_noise_std=0.02):
    """
    Apply data augmentation to a training batch.

    Two strategies are applied:
    1. Gaussian noise added to node features to prevent overfitting
       to specific feature magnitudes.
    2. Random edge dropout to prevent the model from relying on any
       fixed subset of inter-channel connections.

    Augmentation is only applied during training — call this function
    only inside the training loop, not during validation or testing.

    Parameters
    ----------
    batch : torch_geometric.data.Batch
        A batched graph object.
    edge_drop_prob : float
        Probability of dropping each edge independently (default: 0.15).
    feat_noise_std : float
        Standard deviation of Gaussian noise added to node features (default: 0.02).

    Returns
    -------
    torch_geometric.data.Batch
        Augmented batch (original is not modified).
    """
    # work on a shallow copy to avoid modifying the original batch
    batch = copy.copy(batch)
    batch.x = batch.x.clone()
    batch.edge_index = batch.edge_index.clone()

    # --- 1. feature noise ---
    if feat_noise_std > 0:
        noise = torch.randn_like(batch.x) * feat_noise_std
        batch.x = batch.x + noise.to(batch.x.device)

    # --- 2. edge dropout ---
    if edge_drop_prob > 0:
        E = batch.edge_index.shape[1]
        # randomly mask edges with probability edge_drop_prob
        mask = torch.rand(E, device=batch.edge_index.device) > edge_drop_prob
        batch.edge_index = batch.edge_index[:, mask]

    return batch



def normalize_with_train_stats(batch, mean, std):
    """
    Normalize node features using statistics computed from the training set.

    Applies the same mean and std to training, validation, and test batches,
    ensuring no information from held-out data influences normalization.

    Parameters
    ----------
    batch : torch_geometric.data.Batch
        A batched graph object.
    mean : torch.Tensor
        Per-feature mean computed from training graphs, shape (n_features,).
    std : torch.Tensor
        Per-feature std computed from training graphs, shape (n_features,).

    Returns
    -------
    torch_geometric.data.Batch
        Normalized batch (original is not modified).
    """
    batch = copy.copy(batch)
    batch.x = batch.x.clone()

    # move statistics to the same device as the batch
    mean = mean.to(batch.x.device)
    std  = std.to(batch.x.device)

    batch.x = (batch.x - mean) / std

    return batch


In [8]:
def run_fold(train_data, train_labels, val_data, val_labels, test_data, test_labels):
    """
    Train and evaluate the model for a single LOSO fold.

    Builds graphs from the input arrays, trains the model with early stopping,
    and evaluates the best checkpoint on the test subject.

    Parameters
    ----------
    train_data : np.ndarray
        EEG features for training subjects, shape (N_train, C, W, B).
    train_labels : np.ndarray
        Labels for training subjects, shape (N_train,).
    val_data : np.ndarray
        EEG features for the validation subject, shape (N_val, C, W, B).
    val_labels : np.ndarray
        Labels for the validation subject, shape (N_val,).
    test_data : np.ndarray
        EEG features for the test subject, shape (N_test, C, W, B).
    test_labels : np.ndarray
        Labels for the test subject, shape (N_test,).

    Returns
    -------
    test_acc : float
        Accuracy on the test subject using the best validation checkpoint.
    model : EEGGAT
        Trained model with best validation weights restored.
    test_graphs : list of torch_geometric.data.Data
        Graph objects built from the test subject data.
    """
    # build graph objects for each split
    train_graphs = build_graphs(train_data, train_labels)
    val_graphs   = build_graphs(val_data, val_labels)
    test_graphs  = build_graphs(test_data, test_labels)

    
    # compute normalization statistics from training graphs only
    # the same statistics are applied to val and test to prevent leakage
    train_mean, train_std = compute_train_stats(train_graphs)

    # --- model setup ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    in_channels = train_graphs[0].x.shape[1]

    model = EEGGAT(
        in_channels,
        hidden_channels,
        NUM_CLASSES,
        dropout=0.4
    ).to(device)


    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # cross-entropy with label smoothing to prevent overconfident predictions
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    # reduce learning rate when validation accuracy stops improving
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=6
    )

    train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_graphs, batch_size=batch_size)
    test_loader  = DataLoader(test_graphs, batch_size=batch_size)

    # --- early stopping state ---
    best_val_acc = 0.0
    best_weights = copy.deepcopy(model.state_dict())  
    patience_counter = 0

    # =========================================================
    # TRAINING LOOP
    # =========================================================
    for epoch in range(1, num_epochs + 1):

        model.train()
        correct_train = 0
        total_train = 0

        for batch in train_loader:
            batch = batch.to(device)
            # normalize using training statistics
            batch = normalize_with_train_stats(batch, train_mean, train_std)
            # apply data augmentation (noise + edge dropout)
            batch = augment_batch(batch, edge_drop_prob, feat_noise_std)

            optimizer.zero_grad()
            out = model(batch)
            loss = criterion(out, batch.y)
            loss.backward()

            # gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            preds = out.argmax(dim=1)
            correct_train += (preds == batch.y).sum().item()
            total_train += batch.num_graphs

        train_acc = correct_train / total_train

        # =========================================================
        # VALIDATION
        # =========================================================
        model.eval()
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                batch = normalize_with_train_stats(batch, train_mean, train_std)

                out = model(batch)
                preds = out.argmax(dim=1)

                correct_val += (preds == batch.y).sum().item()
                total_val += batch.num_graphs

        val_acc = correct_val / total_val

        # step scheduler based on validation accuracy
        scheduler.step(val_acc)

        # =========================================================
        # EARLY STOPPING
        # =========================================================
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break



    # =========================================================
    # FINAL TEST EVALUATION
    # =========================================================
    # restore the checkpoint with the highest validation accuracy
    model.load_state_dict(best_weights)
    model.eval()

    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            batch = normalize_with_train_stats(batch, train_mean, train_std)

            out = model(batch)
            preds = out.argmax(dim=1)

            correct_test += (preds == batch.y).sum().item()
            total_test += batch.num_graphs

    test_acc = correct_test / total_test

    return test_acc, model, test_graphs

In [9]:
# =========================================================
# CHANNEL NAMES
# reverse mapping: index -> channel name, for display purposes
# =========================================================
ind_for_channel = {v: k for k, v in canais_ind.items()}
channel_names = [ind_for_channel[i] for i in channels_selected]

print("\nChannels used in the model:")
print(channel_names)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# METRICS
# =========================================================
all_test_acc = []
n_subjects = len(subject_data)

# =========================================================
# HYPERPARAMETERS
# =========================================================
hidden_channels = 64        # hidden dimension for GAT layers and classifier
NUM_CLASSES     = 3         # number of emotion classes (negative, neutral, positive)
lr              = 1e-3      # initial learning rate for AdamW
weight_decay    = 1e-3      # L2 regularization strength
batch_size      = 16        # number of graphs per training batch
num_epochs      = 50        # maximum number of training epochs per fold
patience        = 10        # early stopping patience (epochs without improvement)
edge_drop_prob  = 0.15      # probability of dropping each edge during augmentation
feat_noise_std  = 0.01      # std of Gaussian noise added to node features
label_smoothing = 0.15      # label smoothing factor for cross-entropy loss
grad_clip       = 1.0       # maximum gradient norm for clipping



# =========================================================
# LOOP LOSO
# =========================================================
for test_subject in range(n_subjects):

    print("\n==============================")
    print(f"TEST SUBJECT {test_subject}")
    print("------------------------------")

    # --- test set: held-out subject ---
    test_data = subject_data[test_subject]
    test_labels = subject_labels[test_subject]

    # --- validation set: deterministically selected from remaining subjects ---
    # subject at position (test_subject % 14) in the list of remaining subjects
    remaining = [i for i in range(n_subjects) if i != test_subject]
    val_subject = remaining[test_subject % len(remaining)]

    val_data = subject_data[val_subject]
    val_labels = subject_labels[val_subject]

    # --- training set: all remaining subjects except validation ---
    train_subjects = [i for i in remaining if i != val_subject]
    train_data = np.concatenate([subject_data[i] for i in train_subjects])
    train_labels = np.concatenate([subject_labels[i] for i in train_subjects])

    
    # =========================================================
    # TRAINING
    # =========================================================
    test_acc, model, test_graphs = run_fold(
        train_data, train_labels,
        val_data, val_labels,
        test_data, test_labels
    )

    print(f"Test accuracy subject {test_subject}: {test_acc:.4f}")
    all_test_acc.append(test_acc)

# =========================================================
# FINAL RESULTS
# =========================================================
all_test_acc = np.array(all_test_acc)


print("\n====================================")
print("FINAL RESULTS (LOSO)")
print("====================================")
print(f"Mean:  {all_test_acc.mean():.4f}")
print(f"Std:   {all_test_acc.std():.4f}")
print(f"Min:   {all_test_acc.min():.4f}")
print(f"Max:   {all_test_acc.max():.4f}")

print("\nAccuracy per subject:")
for i, acc in enumerate(all_test_acc):
    print(f"  Subject {i}: {acc:.4f}")


Channels used in the model:
['FT7', 'FT8', 'P7', 'P8']

TEST SUBJECT 0
------------------------------
Test accuracy subject 0: 0.9556

TEST SUBJECT 1
------------------------------
Test accuracy subject 1: 0.9333

TEST SUBJECT 2
------------------------------
Test accuracy subject 2: 1.0000

TEST SUBJECT 3
------------------------------
Test accuracy subject 3: 0.9333

TEST SUBJECT 4
------------------------------
Test accuracy subject 4: 0.9333

TEST SUBJECT 5
------------------------------
Test accuracy subject 5: 0.8444

TEST SUBJECT 6
------------------------------
Test accuracy subject 6: 0.9333

TEST SUBJECT 7
------------------------------
Test accuracy subject 7: 1.0000

TEST SUBJECT 8
------------------------------
Test accuracy subject 8: 0.9556

TEST SUBJECT 9
------------------------------
Test accuracy subject 9: 0.9333

TEST SUBJECT 10
------------------------------
Test accuracy subject 10: 1.0000

TEST SUBJECT 11
------------------------------
Test accuracy subject 11: